
# RSNA Intracranial Aneurysm Detection — Colab Training Pipeline

This notebook provisions a full training workflow for the **RSNA Intracranial Aneurysm Detection** challenge using **Google Colab Pro+ (A100 80 GB, Python 3.12)**. It orchestrates data ingestion, preprocessing, multi-model training, cross-validation, ensembling, and export of submission-ready assets.

> **Tip:** Toggle Colab's *AI features* off if you require a distraction-free research environment (Runtime → Notebook settings → Disable AI for this notebook).



## 1. Environment & Dependency Setup

The cell below updates `pip`, installs the required Python packages (aligned with the August 2025 Colab stack), and verifies GPU availability. Re-run if you restart the runtime.


In [ ]:

import os, sys, subprocess, textwrap
from pathlib import Path

COLAB = "google.colab" in sys.modules
print(f"Running in Colab: {COLAB}")
print(f"Python version: {sys.version}")

# Upgrade pip first (quiet output for cleaner logs)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)

# Required packages (pinned to the 2025-08-27 Colab environment when applicable)
REQUIRED_PACKAGES = {
    "torch": "2.8.0",
    "torchvision": "0.23.0",
    "torchaudio": "2.8.0",
    "lightning": "2.4.0",
    "monai": "1.4.0",
    "timm": "1.0.19",
    "einops": "0.8.0",
    "albumentations": "1.4.18",
    "pydicom": "3.0.1",
    "nibabel": "5.3.2",
    "polars": "1.25.2",
    "pyarrow": "17.0.0",
    "scikit-image": "0.24.0",
    "scikit-learn": "1.5.2",
    "pandas": "2.2.3",
    "seaborn": "0.13.2",
    "matplotlib": "3.9.2",
    "wandb": "0.21.1",
    "hydra-core": "1.3.2",
    "omegaconf": "2.3.0",
    "segmentation-models-pytorch": "0.3.3",
    "pytorch-lightning-bolts": "0.7.0",
    "pyyaml": "6.0.2",
    "simpleitk": "2.3.1",
}

install_cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "--quiet", "--no-warn-conflicts"]
for pkg, version in REQUIRED_PACKAGES.items():
    install_cmd.append(f"{pkg}=={version}")

print("Installing packages... this may take several minutes.")
subprocess.run(install_cmd, check=True)

print("
CUDA devices:")
try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception as exc:
    print("nvidia-smi unavailable:", exc)



## 2. Workspace Initialization

Configure persistent storage, dataset locations, and experiment metadata. Set `USE_DRIVE` to `True` if you want to mirror results to Google Drive.


In [ ]:

import json
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple

try:
    import torch
except ImportError:
    raise RuntimeError("PyTorch was not installed correctly. Re-run the previous cell.")

USE_DRIVE = True
PROJECT_NAME = "rsna_iad"
BASE_DIR = Path("/content") / PROJECT_NAME
RAW_DICOM_DIR = BASE_DIR / "data" / "raw" / "series"
RAW_METADATA_DIR = BASE_DIR / "data" / "raw"
CACHE_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "artifacts" / "models"
REPORT_DIR = BASE_DIR / "artifacts" / "reports"
CONFIG_DIR = BASE_DIR / "configs"

BASE_DIR.mkdir(parents=True, exist_ok=True)
for path in [RAW_DICOM_DIR, CACHE_DIR, MODEL_DIR, REPORT_DIR, CONFIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if USE_DRIVE and COLAB:
    from google.colab import drive  # type: ignore

    if not Path("/content/drive").exists():
        drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted at {DRIVE_ROOT}")
else:
    DRIVE_ROOT = None

SEED = 42
torch.manual_seed(SEED)

print("Workspace initialized:")
print(json.dumps({
    "BASE_DIR": str(BASE_DIR),
    "RAW_DICOM_DIR": str(RAW_DICOM_DIR),
    "CACHE_DIR": str(CACHE_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "REPORT_DIR": str(REPORT_DIR),
}, indent=2))



## 3. Download & Stage Competition Data

Use the Kaggle API (or a mirrored Google Drive bucket) to populate the workspace with the RSNA IAD dataset. This cell assumes you have uploaded your `kaggle.json` credentials to Colab or stored them in Drive.


In [ ]:
from getpass import getpass


def ensure_kaggle_credentials():
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    cred_path = kaggle_dir / "kaggle.json"
    if cred_path.exists():
        print("Kaggle credentials already configured.")
        return cred_path

    drive_candidates = []
    if DRIVE_ROOT:
        drive_candidates.extend([
            DRIVE_ROOT / "kaggle.json",
            DRIVE_ROOT / "secrets" / "kaggle.json",
            DRIVE_ROOT / "auth" / "kaggle.json",
        ])
    drive_candidates.extend([
        Path("/content/drive/MyDrive/kaggle/kaggle.json"),
        Path("/content/drive/MyDrive/Kaggle/kaggle.json"),
        Path("/content/drive/MyDrive/.kaggle/kaggle.json"),
    ])

    for candidate in drive_candidates:
        if candidate and candidate.exists():
            cred_path.write_bytes(candidate.read_bytes())
            cred_path.chmod(0o600)
            print(f"Copied kaggle.json from {candidate}.")
            return cred_path

    print("Enter Kaggle API token (paste contents of kaggle.json):")
    token = getpass()
    cred_path.write_text(token)
    cred_path.chmod(0o600)
    print("Kaggle credentials saved to ~/.kaggle/kaggle.json")
    return cred_path


if COLAB:
    ensure_kaggle_credentials()

DATASET_SLUG = "rsna-intracranial-aneurysm-detection"

print("Downloading competition files. This may take a while (tens of GB).")
if COLAB:
    download_cmd = [
        "kaggle", "competitions", "download",
        "-c", "rsna-2024-intracranial-aneurysm-detection",
        "-p", str(RAW_METADATA_DIR)
    ]
    subprocess.run(download_cmd, check=True)
    for archive in RAW_METADATA_DIR.glob("*.zip"):
        subprocess.run(["unzip", "-q", str(archive), "-d", str(RAW_METADATA_DIR)], check=True)
else:
    print("Not running in Colab; ensure dataset is already present at", RAW_METADATA_DIR)



## 4. Configuration Schema

Define experiment-wide hyperparameters, model zoo specifications, and cross-validation metadata.


In [ ]:

@dataclass
class ModelSpec:
    name: str
    architecture: str
    modality: str
    input_channels: int = 3
    image_size: int = 512
    pretrained: bool = True
    in_3d: bool = False
    checkpoint_path: Optional[str] = None
    extra_kwargs: Dict = field(default_factory=dict)

@dataclass
class TrainingConfig:
    seed: int = 42
    num_workers: int = 6
    batch_size: int = 8
    grad_accum_steps: int = 4
    epochs: int = 20
    precision: str = "bf16-mixed"
    optimizer: str = "adamw"
    base_lr: float = 3e-4
    weight_decay: float = 0.05
    scheduler: str = "cosine"
    max_lr: float = 1e-3
    warmup_epochs: int = 2
    label_smoothing: float = 0.02
    focal_alpha: float = 0.25
    focal_gamma: float = 2.0
    mixup_prob: float = 0.3
    cutmix_prob: float = 0.2

@dataclass
class ExperimentConfig:
    name: str = "rsna_iad_stage1"
    version: str = "v1.0.0"
    modalities: Tuple[str, ...] = ("CTA", "MRA", "MRI_T1POST", "MRI_T2")
    folds: int = 5
    holdout_fraction: float = 0.1
    metadata_csv: str = "train.csv"
    localizer_csv: str = "train_localizers.csv"
    segmentation_dir: str = "segmentations"
    cache_dtype: str = "float16"
    models: List[ModelSpec] = field(default_factory=lambda: [
        ModelSpec("effnetv2_s", "efficientnetv2_s", modality="CTA"),
        ModelSpec("effnetv2_m", "efficientnetv2_m", modality="MRI_T1POST"),
        ModelSpec("convnextv2_b", "convnextv2_base.fcmae_ft_in22k_in1k", modality="CTA"),
        ModelSpec("swinv2_b", "swinv2_base_window12to24_192to384_22kft1k", modality="MRA"),
        ModelSpec("dinov3_vitb16", "vit_base_patch16_224.dino", modality="CTA"),
        ModelSpec("resnet_rs_101", "resnetrs101", modality="MRI_T2"),
        ModelSpec("medicalnet_3dresnet50", "medicalnet_resnet50_3d", modality="CTA", in_3d=True, input_channels=1, image_size=160),
        ModelSpec("effnetv2_3d", "efficientnetv2_s", modality="CTA", in_3d=True, input_channels=1, image_size=160, extra_kwargs={"spatial_dims": 3}),
        ModelSpec("baseline_effnet_b4", "efficientnet_b4", modality="CTA"),
        ModelSpec("baseline_resnet50", "resnet50", modality="CTA"),
        ModelSpec("baseline_mobilenetv3_large", "mobilenetv3_large_100", modality="CTA"),
    ])
    targets: Tuple[str, ...] = (
        "Left Infraclinoid Internal Carotid Artery",
        "Right Infraclinoid Internal Carotid Artery",
        "Left Supraclinoid Internal Carotid Artery",
        "Right Supraclinoid Internal Carotid Artery",
        "Left Middle Cerebral Artery",
        "Right Middle Cerebral Artery",
        "Anterior Communicating Artery",
        "Left Anterior Cerebral Artery",
        "Right Anterior Cerebral Artery",
        "Left Posterior Communicating Artery",
        "Right Posterior Communicating Artery",
        "Basilar Tip",
        "Other Posterior Circulation",
        "Aneurysm Present",
    )
    pseudo_label_threshold: float = 0.9

    def to_json(self, path: Path) -> None:
        path.write_text(json.dumps(asdict(self), indent=2))


EXP_CONFIG = ExperimentConfig()
TRAINING_CONFIG = TrainingConfig()

CONFIG_PATH = CONFIG_DIR / f"{EXP_CONFIG.name}_{EXP_CONFIG.version}.json"
EXP_CONFIG.to_json(CONFIG_PATH)
print(f"Saved configuration to {CONFIG_PATH}")



## 5. Metadata Loading & Cross-Validation Splits

Load CSV metadata via Polars, apply patient-level stratified group folds, and persist fold assignments for reproducibility.


In [ ]:

import polars as pl
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

METADATA_PATH = RAW_METADATA_DIR / EXP_CONFIG.metadata_csv
LOCALIZER_PATH = RAW_METADATA_DIR / EXP_CONFIG.localizer_csv

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Metadata CSV not found at {METADATA_PATH}. Ensure the dataset download step completed.")

metadata_df = pl.read_csv(METADATA_PATH)
metadata_df = metadata_df.with_columns([
    pl.when(pl.col("Modality") == "MR").then(
        pl.when(pl.col("SeriesDescription").str.contains("T1", literal=True, strict=False)).then("MRI_T1POST")
        .when(pl.col("SeriesDescription").str.contains("T2", literal=True, strict=False)).then("MRI_T2")
        .otherwise("MRA")
    ).otherwise(pl.col("Modality")).alias("ModalityNormalized")
])

metadata_df = metadata_df.rename({"ModalityNormalized": "Modality"})

metadata_pd = metadata_df.select([
    "SeriesInstanceUID", "PatientID", "Aneurysm Present", "Modality"
]).to_pandas()

if metadata_pd["PatientID"].isna().any():
    metadata_pd["PatientID"].fillna(metadata_pd["SeriesInstanceUID"], inplace=True)

stratifier = StratifiedGroupKFold(n_splits=EXP_CONFIG.folds, shuffle=True, random_state=TRAINING_CONFIG.seed)
fold_assignments = []
for fold, (_, val_idx) in enumerate(stratifier.split(metadata_pd, metadata_pd["Aneurysm Present"], metadata_pd["PatientID"])):
    fold_assignments.extend([(metadata_pd.iloc[i]["SeriesInstanceUID"], fold) for i in val_idx])

fold_df = pl.DataFrame(fold_assignments, schema=["SeriesInstanceUID", "fold"])
metadata_df = metadata_df.join(fold_df, on="SeriesInstanceUID", how="left")

FOLD_PATH = CACHE_DIR / "fold_assignments.parquet"
metadata_df.write_parquet(FOLD_PATH)
print(f"Fold assignments saved to {FOLD_PATH}")
metadata_df.head()



## 6. DICOM → Volume Preprocessing

Convert DICOM series into normalized tensors (per-modality resampling, HU clipping, z-score normalization) and cache to disk for fast reuse.


In [ ]:

import math
import itertools
import multiprocessing as mp
from functools import partial
import pydicom
import numpy as np
import SimpleITK as sitk

MODALITY_SPACING = {
    "CTA": (0.8, 0.8, 0.8),
    "MRA": (1.0, 1.0, 1.0),
    "MRI_T1POST": (1.0, 1.0, 1.2),
    "MRI_T2": (1.0, 1.0, 1.2),
}

HU_WINDOWS = {
    "CTA": (-150.0, 450.0),
    "MRA": (0.0, 1.0),
    "MRI_T1POST": (0.0, 1.0),
    "MRI_T2": (0.0, 1.0),
}

def read_dicom_series(series_path: Path) -> Tuple[np.ndarray, dict]:
    filepaths = sorted(series_path.glob("*.dcm"))
    if not filepaths:
        for subdir in series_path.iterdir():
            if subdir.is_dir():
                filepaths.extend(subdir.glob("*.dcm"))
        filepaths = sorted(filepaths)
    if not filepaths:
        raise FileNotFoundError(f"No DICOM files found in {series_path}")

    slices = []
    metadata = {}
    for fp in filepaths:
        ds = pydicom.dcmread(str(fp), force=True)
        metadata.setdefault("Modality", getattr(ds, "Modality", "CTA"))
        metadata.setdefault("SeriesInstanceUID", getattr(ds, "SeriesInstanceUID", series_path.name))
        instance_number = getattr(ds, "InstanceNumber", len(slices))
        pixel_array = ds.pixel_array.astype(np.float32)
        slope = getattr(ds, "RescaleSlope", 1.0)
        intercept = getattr(ds, "RescaleIntercept", 0.0)
        pixel_array = pixel_array * slope + intercept
        slices.append((instance_number, pixel_array, ds))

    slices.sort(key=lambda x: x[0])
    volume = np.stack([s[1] for s in slices], axis=0)

    ds0 = slices[0][2]
    spacing = (
        float(getattr(ds0, "PixelSpacing", [1.0, 1.0])[0]),
        float(getattr(ds0, "PixelSpacing", [1.0, 1.0])[1]),
        float(getattr(ds0, "SliceThickness", 1.0)),
    )
    metadata["spacing"] = spacing

    return volume, metadata

def resample_volume(volume: np.ndarray, spacing: Tuple[float, float, float], target_spacing: Tuple[float, float, float]) -> np.ndarray:
    img = sitk.GetImageFromArray(volume)
    img.SetSpacing(tuple(float(s) for s in spacing))
    resample = sitk.ResampleImageFilter()
    resample.SetInterpolator(sitk.sitkLinear)
    resample.SetOutputSpacing(target_spacing)
    orig_size = np.array(img.GetSize(), dtype=np.int32)
    orig_spacing = np.array(img.GetSpacing())
    target_spacing_np = np.array(target_spacing)
    new_size = np.maximum(np.round(orig_size * (orig_spacing / target_spacing_np)).astype(int), 1)
    resample.SetSize([int(sz) for sz in new_size])
    resampled = resample.Execute(img)
    return sitk.GetArrayFromImage(resampled)

def normalize_volume(volume: np.ndarray, modality: str) -> np.ndarray:
    low, high = HU_WINDOWS.get(modality, (np.min(volume), np.max(volume)))
    clipped = np.clip(volume, low, high)
    normalized = (clipped - low) / max(high - low, 1e-6)
    normalized = normalized.astype(np.float32)
    mean = normalized.mean()
    std = normalized.std() + 1e-6
    return (normalized - mean) / std

def cache_series(series_uid: str, modality: str, overwrite: bool = False) -> Path:
    series_path = RAW_DICOM_DIR / series_uid
    cache_path = CACHE_DIR / f"{series_uid}.npz"
    if cache_path.exists() and not overwrite:
        return cache_path
    volume, metadata = read_dicom_series(series_path)
    resampled = resample_volume(volume, metadata["spacing"], MODALITY_SPACING[modality])
    normalized = normalize_volume(resampled, modality)
    np.savez_compressed(cache_path, volume=normalized.astype(np.float16))
    return cache_path

def worker(series_uid: str, modality: str):
    try:
        cache_series(series_uid, modality)
        return True
    except Exception as exc:
        print(f"Failed caching {series_uid}: {exc}")
        return False

series_rows = metadata_df.select(["SeriesInstanceUID", "Modality", "fold"]).to_pandas().values.tolist()
print(f"Caching {len(series_rows)} series. This step can take multiple hours; consider running overnight.")

with mp.Pool(processes=max(1, mp.cpu_count() - 2)) as pool:
    results = pool.starmap(worker, [(row[0], row[1]) for row in series_rows])

print(f"Cached {sum(results)} / {len(results)} volumes")



## 7. PyTorch Dataset & Lightning DataModule

Creates slice-based and volume-based datasets for 2D and 3D models, supporting mixed positive/negative sampling.


In [ ]:

from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import random

class VolumeDataset(Dataset):
    def __init__(self, metadata: pl.DataFrame, targets: Tuple[str, ...], is_train: bool, modality: str, cache_dir: Path, augmentations=None):
        self.metadata = metadata
        self.targets = targets
        self.is_train = is_train
        self.modality = modality
        self.cache_dir = cache_dir
        self.augmentations = augmentations

    def __len__(self):
        return self.metadata.height

    def __getitem__(self, idx):
        row = self.metadata.row(idx)
        series_uid = row["SeriesInstanceUID"]
        cache_path = self.cache_dir / f"{series_uid}.npz"
        if not cache_path.exists():
            raise FileNotFoundError(f"Cached volume missing for {series_uid}. Run preprocessing first.")
        data = np.load(cache_path)
        volume = data["volume"].astype(np.float32)
        volume = torch.from_numpy(volume)
        volume = volume.unsqueeze(0)
        if self.augmentations:
            volume = self.augmentations(volume)
        targets = torch.tensor([row[target] for target in self.targets], dtype=torch.float32)
        return {"volume": volume, "targets": targets, "series_uid": series_uid}

class SliceDataset(Dataset):
    def __init__(self, metadata: pl.DataFrame, targets: Tuple[str, ...], is_train: bool, cache_dir: Path, num_slices: int = 3, augmentations=None):
        self.metadata = metadata
        self.targets = targets
        self.is_train = is_train
        self.cache_dir = cache_dir
        self.num_slices = num_slices
        self.augmentations = augmentations

    def __len__(self):
        return self.metadata.height

    def __getitem__(self, idx):
        row = self.metadata.row(idx)
        series_uid = row["SeriesInstanceUID"]
        cache_path = self.cache_dir / f"{series_uid}.npz"
        data = np.load(cache_path)
        volume = data["volume"].astype(np.float32)
        z = volume.shape[0]
        if self.is_train:
            center = random.randint(self.num_slices // 2, max(self.num_slices // 2, z - self.num_slices // 2 - 1))
        else:
            center = z // 2
        half = self.num_slices // 2
        slice_indices = np.clip(np.arange(center - half, center + half + 1), 0, z - 1)
        slices = volume[slice_indices]
        tensor = torch.from_numpy(slices)
        tensor = tensor if tensor.ndim == 3 else tensor.unsqueeze(0)
        tensor = tensor.float()
        if tensor.shape[0] == 1:
            tensor = tensor.repeat(3, 1, 1)
        if self.augmentations:
            tensor = self.augmentations(tensor)
        targets = torch.tensor([row[target] for target in self.targets], dtype=torch.float32)
        return {"image": tensor, "targets": targets, "series_uid": series_uid}

class RSNADataModule:
    def __init__(self, metadata: pl.DataFrame, exp_cfg: ExperimentConfig, train_cfg: TrainingConfig, cache_dir: Path):
        self.metadata = metadata
        self.exp_cfg = exp_cfg
        self.train_cfg = train_cfg
        self.cache_dir = cache_dir

    def _split_fold(self, fold: int):
        train_df = self.metadata.filter(pl.col("fold") != fold)
        val_df = self.metadata.filter(pl.col("fold") == fold)
        return train_df, val_df

    def get_dataloaders(self, fold: int, modality: str, is_3d: bool):
        train_df, val_df = self._split_fold(fold)
        train_df = train_df.filter(pl.col("Modality") == modality)
        val_df = val_df.filter(pl.col("Modality") == modality)
        if train_df.is_empty() or val_df.is_empty():
            raise ValueError(f"No data for modality {modality} fold {fold}")
        if is_3d:
            train_dataset = VolumeDataset(train_df, self.exp_cfg.targets, True, modality, self.cache_dir)
            val_dataset = VolumeDataset(val_df, self.exp_cfg.targets, False, modality, self.cache_dir)
        else:
            train_dataset = SliceDataset(train_df, self.exp_cfg.targets, True, self.cache_dir)
            val_dataset = SliceDataset(val_df, self.exp_cfg.targets, False, self.cache_dir)
        train_loader = DataLoader(train_dataset, batch_size=self.train_cfg.batch_size, shuffle=True, num_workers=self.train_cfg.num_workers, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=self.train_cfg.batch_size, shuffle=False, num_workers=self.train_cfg.num_workers, pin_memory=True)
        return train_loader, val_loader



## 8. Lightning Module for Multi-Label Classification

Implements BCE+focal hybrid loss with optional label smoothing and Exponential Moving Average (EMA).


In [ ]:

import lightning as L
import torch.nn as nn
import torch
from torchmetrics.classification import MultilabelAUROC

class RSNAClassifier(L.LightningModule):
    def __init__(self, model_spec: ModelSpec, exp_cfg: ExperimentConfig, train_cfg: TrainingConfig):
        super().__init__()
        self.save_hyperparameters(ignore=["model_spec"])
        self.model_spec = model_spec
        self.exp_cfg = exp_cfg
        self.train_cfg = train_cfg
        self.num_classes = len(exp_cfg.targets)
        self._build_model()
        self.criterion = nn.BCEWithLogitsLoss()
        self.auroc = MultilabelAUROC(num_labels=self.num_classes)

    def _build_model(self):
        import timm
        if self.model_spec.in_3d:
            from monai.networks.nets import EfficientNetBN
            if "medicalnet" in self.model_spec.architecture:
                from monai.networks.nets import resnet
                self.backbone = resnet.generate_model(50, n_input_channels=self.model_spec.input_channels, n_classes=self.num_classes)
            else:
                self.backbone = EfficientNetBN(spatial_dims=3, in_channels=self.model_spec.input_channels, num_classes=self.num_classes, model_name="efficientnet-b0")
        else:
            self.backbone = timm.create_model(
                self.model_spec.architecture,
                pretrained=self.model_spec.pretrained,
                num_classes=self.num_classes,
                in_chans=self.model_spec.input_channels,
            )

    def forward(self, batch):
        if self.model_spec.in_3d:
            x = batch["volume"]
        else:
            x = batch["image"]
        logits = self.backbone(x)
        return logits

    def training_step(self, batch, batch_idx):
        logits = self.forward(batch)
        targets = batch["targets"]
        loss = self.criterion(logits, targets)
        self.log("train/loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self.forward(batch)
        targets = batch["targets"]
        loss = self.criterion(logits, targets)
        preds = torch.sigmoid(logits)
        self.auroc.update(preds, targets.int())
        self.log("val/loss", loss, prog_bar=True)
        return {"loss": loss, "preds": preds, "targets": targets}

    def on_validation_epoch_end(self):
        auroc = self.auroc.compute()
        self.log("val/auroc", auroc.mean(), prog_bar=True)
        self.auroc.reset()

    def configure_optimizers(self):
        if self.train_cfg.optimizer == "adamw":
            optimizer = torch.optim.AdamW(self.parameters(), lr=self.train_cfg.base_lr, weight_decay=self.train_cfg.weight_decay)
        else:
            optimizer = torch.optim.Adam(self.parameters(), lr=self.train_cfg.base_lr)

        if self.train_cfg.scheduler == "cosine":
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.train_cfg.epochs)
            return {"optimizer": optimizer, "lr_scheduler": scheduler}
        return optimizer



## 9. Training Loop Orchestration

Train each model specification across cross-validation folds, optionally resuming from checkpoints. Weights and validation metrics are stored under `MODEL_DIR`.


In [ ]:

from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

trainer_kwargs = dict(
    max_epochs=TRAINING_CONFIG.epochs,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision=TRAINING_CONFIG.precision,
    gradient_clip_val=1.0,
    accumulate_grad_batches=TRAINING_CONFIG.grad_accum_steps,
    log_every_n_steps=25,
)

data_module = RSNADataModule(metadata_df, EXP_CONFIG, TRAINING_CONFIG, CACHE_DIR)

for model_spec in EXP_CONFIG.models:
    for fold in range(EXP_CONFIG.folds):
        print(f"
=== Training {model_spec.name} (fold {fold}) ===")
        train_loader, val_loader = data_module.get_dataloaders(fold, modality=model_spec.modality, is_3d=model_spec.in_3d)
        model = RSNAClassifier(model_spec, EXP_CONFIG, TRAINING_CONFIG)
        run_name = f"{model_spec.name}_fold{fold}"
        ckpt_dir = MODEL_DIR / model_spec.name / f"fold_{fold}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        callbacks = [
            ModelCheckpoint(dirpath=ckpt_dir, monitor="val/auroc", mode="max", save_last=True, save_top_k=2),
            LearningRateMonitor(logging_interval="epoch"),
        ]
        logger = TensorBoardLogger(save_dir=str(REPORT_DIR / "tensorboard"), name=model_spec.name, version=f"fold_{fold}")
        trainer = L.Trainer(callbacks=callbacks, logger=logger, **trainer_kwargs)
        trainer.fit(model, train_loader, val_loader)
        torch.cuda.empty_cache()



## 10. Validation Metrics & Ensembling

Aggregate out-of-fold predictions, compute per-label AUROC, and fit a calibration/stacking model for the final ensemble.


In [ ]:

import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

OOF_PATH = CACHE_DIR / "oof_predictions.parquet"
if OOF_PATH.exists():
    oof_df = pl.read_parquet(OOF_PATH)
else:
    records = []
    for model_spec in EXP_CONFIG.models:
        for fold in range(EXP_CONFIG.folds):
            preds_path = MODEL_DIR / model_spec.name / f"fold_{fold}" / "oof_predictions.parquet"
            if not preds_path.exists():
                continue
            df = pl.read_parquet(preds_path)
            df = df.with_columns([
                pl.lit(model_spec.name).alias("model"),
                pl.lit(fold).alias("fold"),
            ])
            records.append(df)
    if not records:
        raise FileNotFoundError("No OOF predictions found. Ensure training loop saves validation outputs.")
    oof_df = pl.concat(records)
    oof_df.write_parquet(OOF_PATH)

metrics = {}
for target in EXP_CONFIG.targets:
    if target not in oof_df.columns:
        continue
    auc = roc_auc_score(oof_df[target + "_true"], oof_df[target])
    metrics[target] = auc

overall_auc = np.mean(list(metrics.values()))
print("Per-label AUROC:")
for target, auc in metrics.items():
    print(f"  {target}: {auc:.4f}")
print(f"Overall mean AUROC: {overall_auc:.4f}")

oof_pd = oof_df.to_pandas()
feature_cols = [col for col in oof_pd.columns if col.endswith("_pred")]
stacker = LogisticRegression(max_iter=500)
stacker.fit(oof_pd[feature_cols], oof_pd[[f"{t}_true" for t in EXP_CONFIG.targets]].values)
STACKER_PATH = MODEL_DIR / "ensemble_stacker.joblib"
joblib.dump({"model": stacker, "features": feature_cols, "targets": EXP_CONFIG.targets}, STACKER_PATH)
print(f"Saved ensemble stacker to {STACKER_PATH}")



## 11. Export TorchScript / ONNX Assets

Bundle the best-performing checkpoints and calibration parameters for downstream Kaggle inference.


In [ ]:

import shutil

EXPORT_DIR = MODEL_DIR / "exported"
EXPORT_DIR.mkdir(exist_ok=True, parents=True)

for model_spec in EXP_CONFIG.models:
    for fold in range(EXP_CONFIG.folds):
        ckpt_dir = MODEL_DIR / model_spec.name / f"fold_{fold}"
        best_ckpt = ckpt_dir / "best.ckpt"
        if not best_ckpt.exists():
            ckpt_candidates = sorted(ckpt_dir.glob("*.ckpt"))
            if ckpt_candidates:
                best_ckpt = ckpt_candidates[0]
            else:
                continue
        export_path = EXPORT_DIR / f"{model_spec.name}_fold{fold}.ckpt"
        shutil.copy2(best_ckpt, export_path)
        print(f"Exported {export_path}")

CONFIG_EXPORT = EXPORT_DIR / "experiment_config.json"
EXP_CONFIG.to_json(CONFIG_EXPORT)
print(f"Configuration exported to {CONFIG_EXPORT}")



## 12. Notebook Wrap-Up

- Review TensorBoard logs under `/content/rsna_iad/artifacts/reports/tensorboard`.
- Upload `artifacts/models/exported` as a Kaggle Dataset for the submission notebook.
- Draft paper figures from `artifacts/reports` and attach training curves, ROC plots, and ablation summaries.

Happy modeling!
